# 🎙️ LifeMate Master Pipeline — Interactive Testing & Inference
### Zalo AI Challenge 2026 — Track LifeMate (Voice-to-Action Agentic AI)

Notebook này tích hợp toàn bộ hệ sinh thái đã tinh chỉnh:
1. **Tai ASR**: `locvt06/gipformer-finetuned` (Gipformer v4, WER 10.22% / CER 9.21%) & `vinai/PhoWhisper-large`.
2. **Não LLM**: `quanglun/zaic-2026-lifemate-planner-qwen3-4b-lora` (Qwen3-4B LoRA $r=16, \alpha=32$).
3. **Skill Knowledge Graph (Zero-Hardcode)**: Cung cấp 100% công cụ đã cài đặt trên máy người dùng, điều phối ràng buộc Read-before-Write và chống lặp vô tận.
4. **Kaggle Stability & Tối ưu tốc độ**: Fix xung đột `torchao`, tự động merge LoRA trong 30s, stop tokens tức thì trên `</tool_call>`.

> ⚠️ **LƯU Ý CỰC KỲ QUAN TRỌNG VỀ PHẦN CỨNG (ACCELERATOR)**:
> Hãy kiểm tra menu bên phải: **Settings -> Accelerator -> chọn `GPU T4 x2`** (hoặc `GPU P100`).
> Nếu để **CPU**, mô hình 4B sẽ chạy mất **~220s / mẫu**. Khi bật **GPU T4**, tốc độ chỉ còn **~1.5 - 2s / mẫu**!

In [ ]:
# 1. KIỂM TRA GPU VÀ CẢNH BÁO TỐC ĐỘ
import os, torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("=" * 60)
print("🖥️  KIỂM TRA CẤU HÌNH PHẦN CỨNG KAGGLE")
print("=" * 60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_count = torch.cuda.device_count()
    print(f"✅ TÌM THẤY {gpu_count} GPU: {gpu_name}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("🚀 Tốc độ sẵn sàng đạt chuẩn cuộc thi (<= 2 giây / mẫu)!")
else:
    print("❌ CẢNH BÁO: BẠN ĐANG CHẠY TRÊN CPU!")
    print("👉 Hãy nhìn sang menu bên phải: Settings -> Accelerator -> chọn 'GPU T4 x2'!")
    print("👉 Không nên chạy LLM trên CPU vì sẽ mất >200s mỗi câu hỏi.")

print("=" * 60)
!nvidia-smi

In [ ]:
# 2. FIX XUNG ĐỘT TORCHAO & CÀI ĐẶT THƯ VIỆN
# Gỡ torchao để tránh lỗi version compatibility với peft trên Python 3.13 của Kaggle
!pip uninstall -y torchao
!pip install -q -U "transformers>=4.48.0" soundfile accelerate peft trl datasets sherpa-onnx huggingface_hub

In [ ]:
# 3. CLONE HOẶC UPDATE REPOSITORY TỪ GITHUB
import os
REPO_URL = "https://github.com/Quandinh030106/LifeMate-Voice2Action-Agent.git"
REPO_DIR = "/kaggle/working/LifeMate-Voice2Action-Agent"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git fetch origin && git reset --hard origin/main

%cd $REPO_DIR

In [ ]:
# 4. TỰ ĐỘNG DÒ TÌM ĐƯỜNG DẪN CÁC INPUT TRÊN KAGGLE
import glob, os, json

print("=" * 65)
print("🔍 TỰ ĐỘNG DÒ TÌM CÁC ĐƯỜNG DẪN INPUT TRÊN KAGGLE")
print("=" * 65)

# 1. Dò tìm Data Directory (chứa train và public_test)
data_candidates = glob.glob("/kaggle/input/**/public_test.jsonl", recursive=True)
if data_candidates:
    DATA_DIR = os.path.dirname(os.path.dirname(data_candidates[0]))
else:
    DATA_DIR = "/kaggle/input/datasets/locvt06/zaic2026-track-lifemate"
print(f"✅ Data Directory:  {DATA_DIR}")
os.environ["DATA_DIR"] = DATA_DIR

# 2. Dò tìm LoRA Model của bạn Quang Lùn
lora_candidates = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
if lora_candidates:
    LORA_DIR = os.path.dirname(lora_candidates[0])
    with open(lora_candidates[0], "r", encoding="utf-8") as f:
        lora_cfg = json.load(f)
    BASE_MODEL = lora_cfg.get("base_model_name_or_path", "Qwen/Qwen3-4B-Instruct-2507")
    lora_r = lora_cfg.get("r", 16)
    lora_alpha = lora_cfg.get("lora_alpha", 32)
else:
    LORA_DIR = "/kaggle/input/models/quanglun/zaic-2026-lifemate-planner-qwen3-4b-lora/pytorch/epoch-1-peft-adapter/1"
    BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
    lora_r, lora_alpha = 16, 32

print(f"✅ LLM LoRA Path:   {LORA_DIR}")
print(f"   • Base Model:    {BASE_MODEL}")
print(f"   • LoRA Rank:     r={lora_r}, alpha={lora_alpha}")
os.environ["LORA_DIR"] = LORA_DIR
os.environ["BASE_MODEL"] = BASE_MODEL

# 3. Dò tìm Gipformer Fine-tuned của bạn Lộc VT
gip_candidates = glob.glob("/kaggle/input/**/gipformer*/**/selection.json", recursive=True) + glob.glob("/kaggle/input/**/gipformer*/**/model.pt", recursive=True)
if gip_candidates:
    GIP_DIR = os.path.dirname(gip_candidates[0])
else:
    GIP_DIR = "/kaggle/input/datasets/locvt06/gipformer-finetuned"
print(f"✅ Gipformer Path:  {GIP_DIR}")
os.environ["GIP_DIR"] = GIP_DIR
print("=" * 65)

In [ ]:
# 5. XEM CHI TIẾT CẤU HÌNH CỦA BẠN BÈ (LORA & GIPFORMER)
print("=" * 65)
print("📖 XEM THÔNG TIN CỦA CÁC MODEL FINE-TUNED")
print("=" * 65)

# 1. Đọc thông số Gipformer v4
sel_path = os.path.join(GIP_DIR, "selection.json")
if os.path.exists(sel_path):
    print("🎙️  [Gipformer v4 Selection Metrics]:")
    with open(sel_path, "r", encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2))

# 2. Kiểm tra template / cấu hình của LoRA
jinja_path = os.path.join(LORA_DIR, "chat_template.jinja")
if os.path.exists(jinja_path):
    print("\n🧠 [LoRA Chat Template]:")
    with open(jinja_path, "r", encoding="utf-8") as f:
        lines = f.readlines()[:15]
        print("".join(lines) + "...")
print("=" * 65)

In [ ]:
# 6. FAST-MERGE LORA ADAPTER THÀNH STANDALONE MODEL TRONG 30S
# Chạy trong tiến trình OS riêng biệt để tránh lỗi bộ nhớ C-extension của Python 3.13 trên Kaggle
!python -u scripts/merge_lora.py \
    --base_model "$BASE_MODEL" \
    --lora_dir "$LORA_DIR" \
    --output_dir "/kaggle/working/models/lifemate-qwen3-4b-merged"

MERGED_DIR = "/kaggle/working/models/lifemate-qwen3-4b-merged"
os.environ["MERGED_DIR"] = MERGED_DIR

In [ ]:
# 7. [TEST NHANH 3 CÂU] KIỂM TRA CHẾ ĐỘ TEXT TRÊN TRAIN SET (5 GIÂY)
!python -u inference.py \
    --data_dir "$DATA_DIR" \
    --split train \
    --no_audio \
    --limit 3 \
    --llm_model "$MERGED_DIR" \
    --output "/kaggle/working/test_train_3_samples.jsonl"

print("\n" + "=" * 70)
print("🎯 KẾT QUẢ GỌI TOOL TRÊN 3 MẪU TRAIN (CHẾ ĐỘ TEXT):\n")
with open("/kaggle/working/test_train_3_samples.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line.strip())
        print(f"• ID: {row['id']}")
        print(f"  Tool Calls ({len(row['tool_calls'])}): {row['tool_calls']}")
        print(f"  State Hash: {row['state_hash']}\n")
print("=" * 70)

In [ ]:
# 8. [BENCHMARK 100 CÂU TRAIN] ĐÁNH GIÁ ĐỘ CHÍNH XÁC TOÀN DIỆN
# In trực tiếp trên log từng câu:
# - Lời nói ASR nghe được và điểm WER, CER so với Ground Truth
# - Độ chính xác Tool: Đúng mấy trên mấy, Thừa mấy cái, Thiếu mấy cái (P, R, F1)
# - Khớp mã băm State Hash (MATCH / MISMATCH)
# - Điểm số Zalo AI Challenge chính thức

# Ghi chú: Thêm cờ '--no_audio' nếu chỉ muốn test não LLM trên text (mất ~1-2 phút)
# Bỏ cờ '--no_audio' (mặc định) để chạy đầy đủ ASR + LLM (tính WER, CER)

!python -u scripts/benchmark_100_train.py \
    --data_dir "$DATA_DIR" \
    --limit 100 \
    --llm_model "$MERGED_DIR" \
    --asr_engine phowhisper \
    --output "/kaggle/working/benchmark_100_results.jsonl"

In [ ]:
# 9. [TEST 3 CÂU PUBLIC_TEST] KIỂM TRA TOÀN DIỆN AUDIO TRÊN PUBLIC_TEST
!python -u inference.py \
    --data_dir "$DATA_DIR" \
    --split public_test \
    --limit 3 \
    --llm_model "$MERGED_DIR" \
    --asr_engine phowhisper \
    --output "/kaggle/working/test_public_3_samples.jsonl"

print("\n" + "=" * 70)
print("🎯 KẾT QUẢ VOICE-TO-ACTION TRÊN 3 MẪU PUBLIC_TEST (AUDIO THỰC TẾ):\n")
with open("/kaggle/working/test_public_3_samples.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line.strip())
        print(f"• ID: {row['id']}")
        print(f"  Tool Calls ({len(row['tool_calls'])}): {row['tool_calls']}")
        print(f"  State Hash: {row['state_hash']}\n")
print("=" * 70)

In [ ]:
# 10. [CHẠY FULL & NỘP BÀI] KHI BẠN ĐÃ HÀI LÒNG VỚI KẾT QUẢ BENCHMARK
# (Bỏ comment các dòng dưới đây khi bạn muốn chạy toàn bộ 634 mẫu public_test)

# !python -u inference.py \
#     --data_dir "$DATA_DIR" \
#     --split public_test \
#     --llm_model "$MERGED_DIR" \
#     --asr_engine phowhisper \
#     --output "/kaggle/working/submission.jsonl"

# # Kiểm tra và làm sạch submission (đúng chuẩn 634 dòng, chỉ có id, tool_calls, state_hash)
# !python -u scripts/clean_submission.py \
#     --input "/kaggle/working/submission.jsonl" \
#     --data_dir "$DATA_DIR"

# print("\n🎉 TẬP TIN NỘP BÀI ĐÃ SẴN SÀNG TẠI: /kaggle/working/submission.jsonl")